# Opportunity · Tax-Loss Harvesting

Selling a position that's **underwater in a taxable account** realizes a loss you can use
to offset realized gains (and up to \$3,000 of ordinary income), while you stay in the
market via a not-substantially-identical replacement. This finds those lots, sizes the
loss against your gains so far this year, and flags the **wash-sale** trap.

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from invest import config, analysis as A, ledger, mapping, opportunities as O

px.defaults.template = "plotly_white"
PALETTE = px.colors.qualitative.Safe

positions = pd.read_parquet(config.POSITIONS_PARQUET)
history = pd.read_parquet(config.PRICES_PARQUET)
transactions = pd.read_parquet(config.TRANSACTIONS_PARQUET)
entries, _, _ = ledger.load()
account_meta = mapping.load_account_meta()

TODAY = pd.Timestamp.today().normalize()
TOTAL = positions["market_value"].sum()
print(f"${TOTAL:,.0f} · {positions['account_name'].nunique()} accounts · as of {TODAY:%Y-%m-%d}")

## Harvestable losses, by holding

Per-tax-lot, in taxable accounts only, split by **holding term** — a realized loss offsets
same-term gains first (long-term losses → long-term gains), so the split matters.

In [ ]:
lots = O.price_lots(ledger.lots_dataframe(entries), positions)
cand = O.harvest_candidates(lots, account_meta, today=TODAY)
total_loss = cand["unrealized"].sum()
print(f"{len(cand)} loss lots across {cand['symbol'].nunique()} holdings · "
      f"total harvestable loss ${total_loss:,.0f}")

by = cand.groupby(["symbol", "term"])["unrealized"].sum().unstack("term").fillna(0.0)
fig = go.Figure()
for i, term in enumerate([t for t in ("short", "long") if t in by.columns]):
    fig.add_bar(x=by.index, y=by[term], name=f"{term}-term", marker_color=PALETTE[i])
fig.update_layout(barmode="relative", height=420, yaxis_tickprefix="$",
                  title="Harvestable loss by holding & term", hovermode="x unified")
fig.show()

## Offset against this year's realized gains

A harvested loss first nets against gains you've **already realized** in taxable accounts
this year. The waterfall shows where you'd land.

In [ ]:
year = TODAY.year
rg = O.realized_gains_ytd(ledger.realized_gains_dataframe(entries), account_meta, year=year)
realized = rg["realized"].sum()
net = realized + total_loss   # total_loss is negative
fig = go.Figure(go.Waterfall(
    x=[f"Realized gains {year}", "Harvest all losses", "Net taxable gain"],
    measure=["absolute", "relative", "total"],
    y=[realized, total_loss, None],
    text=[f"${realized:,.0f}", f"${total_loss:,.0f}", f"${net:,.0f}"],
    textposition="outside", connector={"line": {"color": "grey"}},
    decreasing={"marker": {"color": "#27ae60"}}, increasing={"marker": {"color": "#c0392b"}},
    totals={"marker": {"color": "#111"}}))
fig.update_layout(height=420, yaxis_tickprefix="$",
                  title=f"Harvest fully offsets {year} gains and leaves ${-net:,.0f} of surplus loss"
                  if net < 0 else f"Net {year} taxable gain after harvesting: ${net:,.0f}")
fig.show()
if net < 0:
    print(f"Surplus ${-net:,.0f} carries against ordinary income (≤$3,000/yr) and forward.")

## Wash-sale check ⚠️

Buying the **same** security within **30 days before or after** a loss sale disallows the
loss. We can see the *before* side now — any candidate you bought recently is flagged
(dividend reinvestment counts as a buy!).

In [ ]:
ws = O.wash_sale_risk(transactions, cand["symbol"].unique(), today=TODAY)
if ws.empty:
    print("✓ No recent purchases of any candidate within the 30-day window.")
else:
    print("⚠ Recent buys — harvesting these risks a wash sale (wait out the window):")
    display(ws.assign(last_buy=lambda d: d["last_buy"].dt.strftime("%Y-%m-%d")))
blocked = set(ws.index)

## The candidate lots — your worklist

Largest losses first. **Clean** lots are clear to harvest; **wash-risk** lots need the
30-day window to pass (or a replacement security). Always confirm against your broker's
own cost basis before trading.

In [ ]:
work = cand.copy()
work["flag"] = np.where(work["symbol"].isin(blocked), "⚠ wash-risk", "clean")
cols = ["account_name", "symbol", "term", "days_held", "quantity",
        "cost_basis", "market_value", "unrealized", "unrealized_pct", "flag"]
(work[cols].head(25).style
   .format({"quantity": "{:.3f}", "cost_basis": "${:,.0f}", "market_value": "${:,.0f}",
            "unrealized": "${:,.0f}", "unrealized_pct": "{:.1%}", "days_held": "{:.0f}"})
   .background_gradient(subset=["unrealized"], cmap="Reds_r"))

---
*These are **candidate generators**, not advice — every row is a starting point for your
own judgment (and, for anything tax-related, a CPA's). Prices are research-grade
(yfinance); cost basis and lots come from the curated ledger, so they're only as right as
its fixups.*
*Wash-sale detection sees only purchases recorded in the ledger and only the backward
window; reinvested dividends and buys in **other** accounts (incl. IRAs) can also trip it.*